# Avance 5.4 - Modelo final

**Proyecto academico:** AURORA - Analisis Unificado de Riesgo Operativo y Readiness con Aprendizaje Automatico  
**Equipo:** 4  
**Modalidad:** Individual  
**Tema:** Ensambles, seleccion del modelo final y evaluacion en datos no vistos

## Aviso de privacidad

Esta libreta usa exclusivamente datos sinteticos. Ninguna fila, identificador, metrica, distribucion o categoria corresponde a datos reales de una organizacion. No se incluyen repositorios, URLs, tecnologias internas, clientes, credenciales, procedimientos operativos ni nombres de sistemas.

## Objetivo del avance

En este avance se comparan modelos de ensamble con el mejor modelo individual obtenido en la fase anterior. El objetivo es determinar si una estrategia de ensamble mejora la capacidad de detectar cambios sinteticos de alto riesgo y elegir un modelo final alineado con el problema.

La evaluacion se organiza en tres partes:

1. Construccion de ensambles homogeneos y heterogeneos.
2. Comparacion contra el mejor modelo individual de Avance 4 usando datos no vistos.
3. Seleccion e interpretacion del modelo final con graficas de rendimiento.

In [1]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    VotingClassifier,
    StackingClassifier,
)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


def find_repo_root(start: Path) -> Path:
    starts = [start, *start.parents]
    for candidate in starts:
        direct = candidate / 'entregas' / 'avance_2_feature_engineering' / 'data' / 'aurora_features_model_ready.csv'
        nested = candidate / 'Repositorio_Tec_AURORA' / 'entregas' / 'avance_2_feature_engineering' / 'data' / 'aurora_features_model_ready.csv'
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / 'Repositorio_Tec_AURORA'
    raise FileNotFoundError('No se encontro el repositorio academico AURORA.')

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / 'entregas' / 'avance_5_modelo_final'
DATA_DIR = NOTEBOOK_DIR / 'data'
FIGURES_DIR = DATA_DIR / 'figures'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DATASET = REPO_ROOT / 'entregas' / 'avance_2_feature_engineering' / 'data' / 'aurora_features_model_ready.csv'
AVANCE4_RESULTS = REPO_ROOT / 'entregas' / 'avance_4_modelos_alternativos' / 'data' / 'aurora_tuned_models_results.csv'

print('Repositorio academico AURORA localizado correctamente.')
print('Dataset sintetico model-ready cargado desde la entrega previa.')

Repositorio academico AURORA localizado correctamente.
Dataset sintetico model-ready cargado desde la entrega previa.


## Carga de datos y separacion train/test

Se utiliza el dataset model-ready construido en Avance 2. La variable objetivo es `high_risk_change`. El conjunto de prueba se reserva para evaluar datos no vistos despues del ajuste de hiperparametros.

In [2]:
df = pd.read_csv(INPUT_DATASET)
TARGET = 'high_risk_change'
leakage_columns = ['change_id', 'failure_category', 'recommended_validation_scope', 'readiness_score']
present_leakage = [column for column in leakage_columns if column in df.columns]

assert TARGET in df.columns, 'No se encontro la variable objetivo.'
assert not present_leakage, f'Columnas con posible fuga de informacion presentes: {present_leakage}'
assert not df.isna().any().any(), 'El dataset model-ready no debe contener valores faltantes.'

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Forma del dataset:', df.shape)
print('Predictores:', X.shape[1])
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Proporcion clase positiva train:', round(y_train.mean(), 4))
print('Proporcion clase positiva test:', round(y_test.mean(), 4))

Forma del dataset: (2400, 61)
Predictores: 60
Train: (1920, 60) Test: (480, 60)
Proporcion clase positiva train: 0.2802
Proporcion clase positiva test: 0.2792


## Metrica principal

La metrica principal es **PR-AUC / Average Precision**, porque la clase positiva representa cambios sinteticos de alto riesgo y es minoritaria. Tambien se reportan ROC-AUC, balanced accuracy, precision, recall, F1, accuracy y tiempo de entrenamiento.

In [3]:
def positive_scores(model, X_values):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_values)[:, 1]
    if hasattr(model, 'decision_function'):
        return model.decision_function(X_values)
    return model.predict(X_values)


def evaluate_model(model, X_values, y_true, model_name, split, training_time_sec=None, model_family=None, ensemble_strategy=None, best_params=None, best_cv_pr_auc=None):
    scores = positive_scores(model, X_values)
    y_pred = model.predict(X_values)
    return {
        'model': model_name,
        'model_family': model_family,
        'ensemble_strategy': ensemble_strategy,
        'split': split,
        'pr_auc_average_precision': average_precision_score(y_true, scores),
        'roc_auc': roc_auc_score(y_true, scores),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_positive': precision_score(y_true, y_pred, zero_division=0),
        'recall_positive': recall_score(y_true, y_pred, zero_division=0),
        'f1_positive': f1_score(y_true, y_pred, zero_division=0),
        'training_time_sec': training_time_sec,
        'best_cv_pr_auc': best_cv_pr_auc,
        'best_params': best_params,
    }

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

## Modelos a comparar

Se incluye el mejor modelo individual de Avance 4 como referencia y seis ensambles:

- Homogeneos: Random Forest, Extra Trees, Gradient Boosting y AdaBoost.
- Heterogeneos: Voting y Stacking.

Los modelos heterogeneos usan los modelos individuales fuertes de la fase anterior, especialmente SVC y regresion logistica, junto con otros clasificadores complementarios.

In [4]:
logistic_prior = LogisticRegression(
    max_iter=4000,
    solver='liblinear',
    C=1.0,
    l1_ratio=1.0,
    class_weight=None,
    random_state=RANDOM_STATE,
)

svc_prior = SVC(
    C=0.1,
    kernel='linear',
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=RANDOM_STATE,
)

knn_prior = KNeighborsClassifier(n_neighbors=15, weights='distance')
gnb_prior = GaussianNB()
mlp_prior = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation='relu',
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    random_state=RANDOM_STATE,
)

model_specs = {
    'individual_svc_tuned': {
        'estimator': svc_prior,
        'param_grid': None,
        'model_family': 'individual_phase_4',
        'ensemble_strategy': 'not_ensemble',
    },
    'random_forest': {
        'estimator': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        'param_grid': {
            'n_estimators': [150, 250],
            'max_depth': [None, 8],
            'min_samples_leaf': [1, 5],
            'class_weight': ['balanced'],
        },
        'model_family': 'ensemble',
        'ensemble_strategy': 'homogeneous_bagging',
    },
    'extra_trees': {
        'estimator': ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        'param_grid': {
            'n_estimators': [150, 250],
            'max_depth': [None, 8],
            'min_samples_leaf': [1, 5],
            'class_weight': ['balanced'],
        },
        'model_family': 'ensemble',
        'ensemble_strategy': 'homogeneous_bagging',
    },
    'gradient_boosting': {
        'estimator': GradientBoostingClassifier(random_state=RANDOM_STATE),
        'param_grid': {
            'n_estimators': [100, 160],
            'learning_rate': [0.03, 0.08],
            'max_depth': [2, 3],
        },
        'model_family': 'ensemble',
        'ensemble_strategy': 'homogeneous_boosting',
    },
    'ada_boost': {
        'estimator': AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            random_state=RANDOM_STATE,
        ),
        'param_grid': {
            'n_estimators': [80, 140],
            'learning_rate': [0.03, 0.10],
        },
        'model_family': 'ensemble',
        'ensemble_strategy': 'homogeneous_boosting',
    },
    'voting_heterogeneous': {
        'estimator': VotingClassifier(
            estimators=[
                ('logistic', logistic_prior),
                ('svc', svc_prior),
                ('knn', knn_prior),
                ('gnb', gnb_prior),
            ],
            voting='soft',
            n_jobs=-1,
        ),
        'param_grid': {
            'weights': [[2, 3, 1, 1], [3, 3, 1, 1], [2, 4, 1, 1], [1, 3, 1, 1]],
        },
        'model_family': 'ensemble',
        'ensemble_strategy': 'heterogeneous_voting',
    },
    'stacking_heterogeneous': {
        'estimator': StackingClassifier(
            estimators=[
                ('logistic', logistic_prior),
                ('svc', svc_prior),
                ('mlp', mlp_prior),
            ],
            final_estimator=LogisticRegression(
                max_iter=3000,
                solver='liblinear',
                class_weight='balanced',
                random_state=RANDOM_STATE,
            ),
            cv=3,
            n_jobs=-1,
            stack_method='auto',
        ),
        'param_grid': {
            'final_estimator__C': [0.1, 1.0, 10.0],
            'passthrough': [False, True],
        },
        'model_family': 'ensemble',
        'ensemble_strategy': 'heterogeneous_stacking',
    },
}

print('Modelos definidos:', list(model_specs))

Modelos definidos: ['individual_svc_tuned', 'random_forest', 'extra_trees', 'gradient_boosting', 'ada_boost', 'voting_heterogeneous', 'stacking_heterogeneous']


## Entrenamiento, ajuste y comparativa

Los ensambles se ajustan con validacion cruzada estratificada sobre el conjunto de entrenamiento. Despues, todos los modelos se evaluan sobre el conjunto de prueba para estimar desempeno en datos no vistos.

In [5]:
fitted_models = {}
comparison_rows = []

for name, spec in model_specs.items():
    estimator = spec['estimator']
    param_grid = spec['param_grid']
    start = time.perf_counter()

    if param_grid:
        search = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring='average_precision',
            cv=cv,
            n_jobs=-1,
            refit=True,
        )
        search.fit(X_train, y_train)
        fitted = search.best_estimator_
        best_params = str(search.best_params_)
        best_cv_pr_auc = search.best_score_
    else:
        fitted = estimator.fit(X_train, y_train)
        best_params = '{}'
        best_cv_pr_auc = np.nan

    elapsed = time.perf_counter() - start
    fitted_models[name] = fitted
    row = evaluate_model(
        fitted,
        X_test,
        y_test,
        model_name=name,
        split='test',
        training_time_sec=elapsed,
        model_family=spec['model_family'],
        ensemble_strategy=spec['ensemble_strategy'],
        best_params=best_params,
        best_cv_pr_auc=best_cv_pr_auc,
    )
    comparison_rows.append(row)
    print(f'{name} entrenado y evaluado.')

comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values(
        by=['pr_auc_average_precision', 'recall_positive', 'balanced_accuracy'],
        ascending=False,
    )
    .reset_index(drop=True)
)
comparison_df.insert(0, 'rank', np.arange(1, len(comparison_df) + 1))
comparison_df.to_csv(DATA_DIR / 'aurora_ensemble_model_comparison.csv', index=False)
comparison_df

individual_svc_tuned entrenado y evaluado.


random_forest entrenado y evaluado.


extra_trees entrenado y evaluado.


gradient_boosting entrenado y evaluado.


ada_boost entrenado y evaluado.


voting_heterogeneous entrenado y evaluado.


stacking_heterogeneous entrenado y evaluado.


,rank,model,model_family,ensemble_strategy,split,pr_auc_average_precision,roc_auc,accuracy,balanced_accuracy,precision_positive,recall_positive,f1_positive,training_time_sec,best_cv_pr_auc,best_params
0,1,stacking_heterogeneous,ensemble,heterogeneous_stacking,test,0.985821,0.994112,0.956250,0.958222,0.889655,0.962687,0.924731,1.691367,0.982480,"{'final_estimator__C': 0.1, 'passthrough': False}"
1,2,individual_svc_tuned,individual_phase_4,not_ensemble,test,0.984393,0.993637,0.952083,0.957618,0.872483,0.970149,0.918728,0.090871,NaN,{}
2,3,voting_heterogeneous,ensemble,heterogeneous_voting,test,0.982542,0.993141,0.964583,0.957143,0.933333,0.940299,0.936803,0.358313,0.982090,"{'weights': [2, 4, 1, 1]}"
3,4,gradient_boosting,ensemble,homogeneous_boosting,test,0.937976,0.974937,0.897917,0.842313,0.897196,0.716418,0.796680,1.958459,0.968716,"{'learning_rate': 0.08, 'max_depth': 2, 'n_est..."
4,5,extra_trees,ensemble,homogeneous_bagging,test,0.921694,0.968445,0.875000,0.787551,0.940476,0.589552,0.724771,1.088887,0.924039,"{'class_weight': 'balanced', 'max_depth': None..."
5,6,random_forest,ensemble,homogeneous_bagging,test,0.910901,0.961910,0.866667,0.772625,0.937500,0.559701,0.700935,4.316907,0.931552,"{'class_weight': 'balanced', 'max_depth': None..."
6,7,ada_boost,ensemble,homogeneous_boosting,test,0.877905,0.946111,0.818750,0.679946,0.960784,0.365672,0.529730,0.738565,0.895579,"{'learning_rate': 0.1, 'n_estimators': 140}"


## Seleccion del modelo final

Se selecciona el modelo final con PR-AUC como criterio principal. En caso de desempenos practicamente empatados, se consideran recall de la clase positiva, balanced accuracy, complejidad e interpretabilidad.

In [6]:
final_model_name = comparison_df.loc[0, 'model']
final_model = fitted_models[final_model_name]
final_scores = positive_scores(final_model, X_test)
final_predictions = final_model.predict(X_test)

final_summary = comparison_df[comparison_df['model'] == final_model_name].copy()
final_summary.to_csv(DATA_DIR / 'aurora_final_model_summary.csv', index=False)

final_predictions_df = pd.DataFrame({
    'row_id': X_test.index,
    'y_true': y_test.values,
    'risk_score': final_scores,
    'y_pred': final_predictions,
})
final_predictions_df.to_csv(DATA_DIR / 'aurora_final_model_predictions.csv', index=False)

print('Modelo final seleccionado:', final_model_name)
print(final_summary[[
    'model',
    'pr_auc_average_precision',
    'roc_auc',
    'balanced_accuracy',
    'precision_positive',
    'recall_positive',
    'f1_positive',
    'training_time_sec',
    'best_params',
]].to_string(index=False))

Modelo final seleccionado: stacking_heterogeneous
                 model  pr_auc_average_precision  roc_auc  balanced_accuracy  precision_positive  recall_positive  f1_positive  training_time_sec                                       best_params
stacking_heterogeneous                  0.985821 0.994112           0.958222            0.889655         0.962687     0.924731           1.691367 {'final_estimator__C': 0.1, 'passthrough': False}


La seleccion final favorece el modelo con mejor PR-AUC en datos no vistos. Si el modelo final es un ensamble heterogeneo, esto sugiere que combinar modelos individuales de buen desempeno puede capturar senales complementarias. En este caso se mantiene especial atencion al recall, porque omitir casos de alto riesgo seria el error mas costoso para el objetivo del caso de estudio.

## Grafica 1: matriz de confusion

La matriz de confusion permite observar aciertos y errores del modelo final separando casos de bajo y alto riesgo. Es util para identificar falsos negativos, que representan cambios sinteticos de alto riesgo que el modelo no detectaria.

In [7]:
cm = confusion_matrix(y_test, final_predictions)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Matriz de confusion - {final_model_name}')
plt.xlabel('Prediccion')
plt.ylabel('Valor real')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_model_confusion_matrix.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/final_model_confusion_matrix.png')
print('TN, FP, FN, TP:', cm.ravel().tolist())

Figura guardada: data/figures/final_model_confusion_matrix.png
TN, FP, FN, TP: [330, 16, 5, 129]


**Interpretacion.** La matriz permite validar si el modelo conserva una deteccion suficiente de la clase positiva. Un numero bajo de falsos negativos indica que el modelo es util para priorizar revisiones de alto riesgo. Los falsos positivos tambien importan, pero son menos criticos que dejar pasar un caso riesgoso sin senalamiento.

## Grafica 2: curva Precision-Recall

La curva Precision-Recall es especialmente relevante porque la clase positiva es minoritaria. Esta visualizacion muestra el intercambio entre detectar mas casos de alto riesgo y mantener precision en las alertas.

In [8]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, final_scores)
pr_auc = average_precision_score(y_test, final_scores)
plt.figure(figsize=(6, 4.5))
plt.plot(recall, precision, label=f'PR-AUC = {pr_auc:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall del modelo final')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_model_precision_recall_curve.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/final_model_precision_recall_curve.png')

Figura guardada: data/figures/final_model_precision_recall_curve.png


**Interpretacion.** Una curva alta indica que el modelo conserva precision incluso cuando aumenta el recall. Para este problema, la curva ayuda a decidir si conviene operar con un umbral mas sensible cuando se quiere reducir el riesgo de omitir cambios importantes.

## Grafica 3: curva ROC

La curva ROC compara la tasa de verdaderos positivos contra la tasa de falsos positivos. Aunque PR-AUC es la metrica principal, ROC-AUC complementa el analisis porque resume separabilidad general entre clases.

In [9]:
fpr, tpr, roc_thresholds = roc_curve(y_test, final_scores)
roc_auc = roc_auc_score(y_test, final_scores)
plt.figure(figsize=(6, 4.5))
plt.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Referencia aleatoria')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC del modelo final')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_model_roc_curve.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/final_model_roc_curve.png')

Figura guardada: data/figures/final_model_roc_curve.png


**Interpretacion.** Una curva cercana al extremo superior izquierdo indica buena separacion entre cambios de bajo y alto riesgo. Si la curva se mantiene por encima de la referencia aleatoria, el modelo aporta una senal util para clasificacion.

## Grafica 4: importancia de caracteristicas por permutacion

La importancia por permutacion mide cuanto cae el desempeno cuando se altera una variable. Esta tecnica es adecuada para comparar variables aun cuando el modelo final sea un ensamble sin coeficientes directos simples.

In [10]:
perm = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring='average_precision',
    n_repeats=6,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)
importance_df.to_csv(DATA_DIR / 'aurora_final_model_feature_importance.csv', index=False)

top_importance = importance_df.head(12)
plt.figure(figsize=(9, 5.5))
sns.barplot(
    data=top_importance,
    x='importance_mean',
    y='feature',
    hue='feature',
    palette='viridis',
    legend=False,
)
plt.title('Importancia por permutacion - modelo final')
plt.xlabel('Caida promedio en PR-AUC')
plt.ylabel('Variable')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_model_permutation_importance.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/final_model_permutation_importance.png')
top_importance

Figura guardada: data/figures/final_model_permutation_importance.png


,feature,importance_mean,importance_std
0,validation_gap_count,0.248372,0.009751
1,prior_failures_90d,0.169965,0.030555
2,release_pressure_score,0.102050,0.009580
3,log1p_modules_touched,0.098600,0.019639
4,critical_files_touched,0.042166,0.007626
5,coverage_delta_pct,0.026558,0.004474
6,log1p_files_changed,0.018468,0.004155
7,defects_per_module,0.014424,0.004486
8,log1p_open_defects_linked,0.013847,0.003854
9,branch_type_development,0.009277,0.003099


**Interpretacion.** Las variables con mayor importancia son aquellas que mas contribuyen a distinguir cambios sinteticos de alto riesgo. Si aparecen senales relacionadas con brechas de evidencia, historial de fallas, complejidad o cobertura, el resultado es consistente con el planteamiento del problema.

## Grafica 5: analisis de umbral

El modelo produce una puntuacion continua de riesgo. El umbral de decision puede ajustarse segun la tolerancia a falsos positivos o falsos negativos. Esta grafica ayuda a analizar ese intercambio.

In [11]:
threshold_values = np.linspace(0.05, 0.95, 19)
threshold_rows = []
for threshold in threshold_values:
    pred = (final_scores >= threshold).astype(int)
    threshold_rows.append({
        'threshold': threshold,
        'precision_positive': precision_score(y_test, pred, zero_division=0),
        'recall_positive': recall_score(y_test, pred, zero_division=0),
        'f1_positive': f1_score(y_test, pred, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred),
    })
threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(DATA_DIR / 'aurora_final_model_threshold_analysis.csv', index=False)

plt.figure(figsize=(8, 5))
plt.plot(threshold_df['threshold'], threshold_df['precision_positive'], marker='o', label='Precision')
plt.plot(threshold_df['threshold'], threshold_df['recall_positive'], marker='o', label='Recall')
plt.plot(threshold_df['threshold'], threshold_df['f1_positive'], marker='o', label='F1')
plt.plot(threshold_df['threshold'], threshold_df['balanced_accuracy'], marker='o', label='Balanced accuracy')
plt.xlabel('Umbral')
plt.ylabel('Metrica')
plt.title('Analisis de umbral del modelo final')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_model_threshold_tradeoff.png', dpi=160, metadata={'Software': 'AURORA synthetic academic analysis'})
plt.close()
print('Figura guardada: data/figures/final_model_threshold_tradeoff.png')
threshold_df.head()

Figura guardada: data/figures/final_model_threshold_tradeoff.png


,threshold,precision_positive,recall_positive,f1_positive,balanced_accuracy
0,0.05,0.279167,1.000000,0.436482,0.500000
1,0.10,0.670000,1.000000,0.802395,0.904624
2,0.15,0.760000,0.992537,0.860841,0.935575
3,0.20,0.815951,0.992537,0.895623,0.952916
4,0.25,0.825000,0.985075,0.897959,0.952075


**Interpretacion.** Un umbral bajo tiende a aumentar recall y reducir falsos negativos, pero puede generar mas falsos positivos. Un umbral alto hace lo contrario. Para una aplicacion de priorizacion de riesgo, el umbral debe elegirse con cuidado segun el costo de omitir casos de alto riesgo.

## Conclusiones

La comparativa muestra que los ensambles heterogeneos pueden mejorar el desempeno frente al modelo individual de referencia, especialmente cuando combinan modelos fuertes de la fase previa. El modelo final se selecciona por su desempeno en datos no vistos, manteniendo un balance razonable entre PR-AUC, recall, balanced accuracy y complejidad.

El resultado no debe interpretarse como una metrica real de una organizacion, ya que los datos son sinteticos. Su valor esta en demostrar un flujo completo y seguro de seleccion de modelo final: preparacion de datos, comparacion de ensambles, ajuste de hiperparametros, evaluacion en test, seleccion argumentada e interpretacion visual.